In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Input, Dropout
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# leading labels
labels_df = pd.read_csv(r"/content/drive/MyDrive/datasets/DAICWOZ/PHQ8_labels.csv")
display(labels_df)

,Participant_ID,PHQ8_Binary,PHQ8_Score
0,300,0,2
1,301,0,3
2,302,0,4
3,303,0,0
4,304,0,6
...,...,...,...
184,488,0,0
185,489,0,3
186,490,0,2
187,491,0,8


In [ ]:
# list of participants
participants = list(labels_df["Participant_ID"])
#participants = [300, 301]
print(participants)

[300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492]


In [ ]:
#print(labels_df[labels_df["Participant_ID"] == 300].iloc[0,2])

2


In [ ]:
def compute_statistics(window):
    stats = {}
    for col in window.columns:
        if window[col].dtype in ['int64', 'float64']:  # Numeric columns
            stats[f'{col}_MinV'] = np.min(window[col], axis = 0)
            stats[f'{col}_MaxV'] = np.max(window[col], axis = 0)
            stats[f'{col}_std'] = np.std(window[col], axis = 0)
            stats[f'{col}_AM'] = np.mean(window[col], axis = 0)
            stats[f'{col}_AAM'] = np.mean(np.abs(window[col]), axis = 0)
            #stats[f'{col}_GM'] = np.exp(np.mean(np.log(window[col] + 1e-9), axis = 0))
            stats[f'{col}_HM'] = len(window[col]) / np.sum(1.0 / (window[col] + 1e-9), axis=0)

            stats[f'{col}_Sum'] = np.sum(window[col], axis=0)
            stats[f'{col}_Q1'] = np.percentile(window[col], 25, axis=0)
            stats[f'{col}_Median'] = np.median(window[col], axis=0)
            stats[f'{col}_Q3'] = np.percentile(window[col], 75, axis=0)
            stats[f'{col}_Variance'] = np.var(window[col], axis=0)
            #stats[f'{col}_Skewness'] = skew(data, axis=0)
            #stats[f'{col}_Kurtosis'] = kurtosis(data, axis=0)
            stats[f'{col}_Zero_Crossings'] = np.sum(np.diff(np.sign(window[col]), axis=0) != 0, axis=0)
            stats[f'{col}_Mean_Energy'] = np.mean(window[col]**2, axis=0)
            stats[f'{col}_Mean_Curve_Length'] = np.mean(np.abs(np.diff(window[col], axis=0)), axis=0)
            #stats[f'{col}_Mean_Teager_Energy'] = np.mean(window[col][1:-1]**2 - window[col][:-2] * window[col][2:], axis=0)

    return pd.Series(stats)


def sliding_window(df, window_size, step_size):
    aggregated_windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        aggregated_window = compute_statistics(window)
        aggregated_windows.append(aggregated_window)
    return pd.DataFrame(aggregated_windows)

In [ ]:
# Load Data
def load_data(participants, labels_df):
    combined_data = {}
    for participant in participants:
        print(participant)
        data = {}
        file_paths = {
                        'CLNF_AUs': rf'/content/drive/MyDrive/datasets/DAICWOZ/video_CLNF/{participant}_P/{participant}_CLNF_AUs.txt',
                        'CLNF_features': rf'/content/drive/MyDrive/datasets/DAICWOZ/video_CLNF/{participant}_P/{participant}_CLNF_features.txt',
                        'CLNF_features3D': rf'/content/drive/MyDrive/datasets/DAICWOZ/video_CLNF/{participant}_P/{participant}_CLNF_features3D.txt',
                        'CLNF_gaze': rf'/content/drive/MyDrive/datasets/DAICWOZ/video_CLNF/{participant}_P/{participant}_CLNF_gaze.txt',
                        'CLNF_pose': rf'/content/drive/MyDrive/datasets/DAICWOZ/video_CLNF/{participant}_P/{participant}_CLNF_pose.txt',
                        }
        for file_type, path in file_paths.items():
                data[file_type] = pd.read_csv(path)

                # droping the zero success rows
                data[file_type].drop(data[file_type].loc[data[file_type][" success"]==0].index, inplace=True)
                data[file_type].reset_index(drop = True, inplace = True)

                data[file_type].drop(columns = [" timestamp", " confidence", " success"], inplace = True)

        merged_data = pd.merge(data['CLNF_features'], data['CLNF_AUs'], on=['frame'])
        merged_data = pd.merge(merged_data, data['CLNF_features3D'], on=['frame'])
        merged_data = pd.merge(merged_data, data['CLNF_gaze'], on=['frame'])
        merged_data = pd.merge(merged_data, data['CLNF_pose'], on=['frame'])
        merged_data.drop(columns = ["frame"], inplace = True)



        window_size = 90  # For 3 second of data, assuming 30 rows per second
        step_size = 60    # Adjust as needed


        #merged_data = sliding_window(merged_data, window_size, step_size)
        # loading target as PHQ_binary
        merged_data["Target"] = labels_df[labels_df["Participant_ID"] == participant].iloc[0,1]



        merged_data.to_csv(f"{participant}_CLNF.csv", index = False)

        #combined_data[participant] = merged_data
    return combined_data

In [ ]:
data = load_data(participants, labels_df)

300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367


<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  data[file_type] = pd.read_csv(path)
<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111

368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
395
396


<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  data[file_type] = pd.read_csv(path)
<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111

397
399
400
401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432


<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  data[file_type] = pd.read_csv(path)
<ipython-input-7-ed141b6316ed>:15: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111

433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
451
452
453
454
455
456
457
458
459
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492


In [15]:
# downloading the csv files
from google.colab import files
#for participant in participants:
#    files.download(fr"/content/{participant}_CLNF.csv")

files.download(fr"/content/315_CLNF.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#display(data[300])

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x_h1,y_h1,z_h1,Tx,Ty,Tz,Rx,Ry,Rz,Target
0,627.755,631.277,638.299,647.668,663.522,686.279,711.835,739.108,767.412,792.931,...,-0.143155,0.428261,-0.892244,69.3181,39.2286,575.033,0.203683,-0.088582,-0.050478,0
1,626.449,630.076,637.020,645.933,661.343,684.353,711.124,739.969,769.517,795.897,...,-0.130209,0.435774,-0.890588,68.8777,38.6533,572.427,0.218002,-0.090140,-0.048800,0
2,626.202,629.321,635.614,643.762,658.157,680.616,708.011,738.505,769.516,796.793,...,-0.124743,0.449668,-0.884442,68.5641,38.2144,570.509,0.229491,-0.095551,-0.043008,0
3,626.005,628.895,634.909,642.659,656.472,678.550,706.125,737.386,769.094,796.817,...,-0.122088,0.459371,-0.879814,68.3127,37.9017,568.938,0.240995,-0.097895,-0.038246,0
4,625.912,628.728,634.655,642.298,655.964,677.949,705.588,737.032,768.951,796.830,...,-0.121251,0.463143,-0.877950,68.2095,37.8085,568.248,0.243577,-0.098172,-0.036744,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19453,544.265,553.300,565.475,578.779,597.998,624.053,653.473,685.938,717.091,744.874,...,0.059400,0.550518,-0.832707,26.8957,78.6713,561.815,0.441858,-0.093022,-0.228571,0
19454,546.264,554.441,566.228,579.684,599.179,624.966,653.216,684.540,715.383,743.123,...,0.026875,0.537094,-0.843094,27.1481,79.1381,561.970,0.430105,-0.082108,-0.212567,0
19455,546.991,554.463,565.709,578.903,597.947,623.281,651.599,683.694,714.814,742.468,...,0.029825,0.550092,-0.834571,27.0585,79.2567,559.403,0.458600,-0.079041,-0.192886,0
19456,548.487,555.796,567.069,580.258,599.205,624.453,652.655,684.236,714.621,741.707,...,0.017142,0.561031,-0.827618,27.8152,79.6674,556.495,0.473191,-0.077877,-0.174793,0


In [17]:
# combining all CLNF files into a single dataframe
complete_video_CLNF = pd.DataFrame()

for participant in participants:
    print(participant)

    temp = pd.read_csv(fr"/content/{participant}_CLNF.csv")
    #print(temp.shape)
    complete_video_CLNF = pd.concat([complete_video_CLNF, temp], ignore_index=True, axis = 0)
    print(complete_video_CLNF.shape)

300
(18981, 379)
301
(43308, 379)
302
(71652, 379)
303
(100723, 379)
304
(121993, 379)
305
(171836, 379)
306
(197103, 379)
307
(230081, 379)
308
(252924, 379)
309
(272564, 379)
310
(295953, 379)
311
(319266, 379)
312
(340538, 379)
313
(362669, 379)
314
(408853, 379)
315
(437600, 379)
316
(463212, 379)
317
(486443, 379)
318
(503809, 379)
319
(522324, 379)
320
(545228, 379)
321
(569502, 379)
322
(600055, 379)
323
(621494, 379)
324
(641199, 379)
325
(667340, 379)
326
(686963, 379)
327
(706723, 379)
328
(738005, 379)
329
(758749, 379)
330
(780151, 379)
331
(805113, 379)
332
(830148, 379)
333
(858923, 379)
334
(887782, 379)
335
(912078, 379)
336
(937824, 379)
337
(993628, 379)
338
(1010926, 379)
339
(1036552, 379)
340
(1054203, 379)
341
(1079593, 379)
343
(1102219, 379)
344
(1128107, 379)
345
(1151228, 379)
346
(1185723, 379)
347
(1203605, 379)
348
(1224962, 379)
349
(1257953, 379)
350
(1283768, 379)
351
(1306187, 379)
352
(1328722, 379)
353
(1351793, 379)
354
(1368963, 379)
355
(1387752, 3

In [20]:
display(complete_video_CLNF.shape)

(5135964, 379)

In [19]:
complete_video_CLNF.to_csv("/content/drive/MyDrive/datasets/DAICWOZ/complete_video_CLNF.csv", index = False)

In [18]:
complete_video_CLNF.to_csv("complete_video_CLNF.csv", index = False)

In [21]:
display(complete_video_CLNF)

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x_h1,y_h1,z_h1,Tx,Ty,Tz,Rx,Ry,Rz,Target
0,627.755,631.277,638.299,647.668,663.522,686.279,711.835,739.108,767.412,792.931,...,-0.143155,0.428261,-0.892244,69.3181,39.2286,575.033,0.203683,-0.088582,-0.050478,0
1,626.449,630.076,637.020,645.933,661.343,684.353,711.124,739.969,769.517,795.897,...,-0.130209,0.435774,-0.890588,68.8777,38.6533,572.427,0.218002,-0.090140,-0.048800,0
2,626.202,629.321,635.614,643.762,658.157,680.616,708.011,738.505,769.516,796.793,...,-0.124743,0.449668,-0.884442,68.5641,38.2144,570.509,0.229491,-0.095551,-0.043008,0
3,626.005,628.895,634.909,642.659,656.472,678.550,706.125,737.386,769.094,796.817,...,-0.122088,0.459371,-0.879814,68.3127,37.9017,568.938,0.240995,-0.097895,-0.038246,0
4,625.912,628.728,634.655,642.298,655.964,677.949,705.588,737.032,768.951,796.830,...,-0.121251,0.463143,-0.877950,68.2095,37.8085,568.248,0.243577,-0.098172,-0.036744,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5135959,706.044,706.535,708.599,713.679,725.442,745.534,770.296,797.526,826.620,853.914,...,-0.220513,0.050259,-0.974088,113.4310,69.3904,604.861,-0.066444,-0.057333,0.002936,0
5135960,706.158,706.808,709.025,714.236,726.229,746.577,771.344,798.425,827.271,854.436,...,-0.216312,0.054710,-0.974790,113.4400,68.3755,603.450,-0.070922,-0.058289,-0.001975,0
5135961,705.604,706.172,708.458,713.864,726.164,746.668,771.379,798.365,827.162,854.324,...,-0.203398,0.044952,-0.978064,113.5900,67.8866,605.178,-0.078595,-0.058826,-0.003926,0
5135962,704.389,705.362,708.076,713.728,725.939,746.269,770.835,798.002,826.924,854.147,...,-0.195401,0.053706,-0.979252,112.9580,68.1986,603.042,-0.073103,-0.059036,-0.004743,0
